# ChemiAI — финальный пайплайн (public best 284.61)

**Автор:** команда ChemiAI

**Дата:** май 2026

**Аннотация:**
Воспроизводимый ноутбук для регрессии по RDKit-дескрипторам.
Собирает сабмит из сырых `train`/`test` без чтения готовых submission CSV.
Лучший public score: **284.60804** (`submission_ic50_size_catboost_w25.csv`).
Ключ: **IC50** = blend clustering + CatBoost(size); **SI** = robust head + инвариант `CC50/IC50`.

## 1. Введение и контекст

Задача — предсказать три таргета для молекул: **IC50**, **CC50**, **SI**.

Метрика соревнования — средний RMSE по трём таргетам.

На train выполняется инвариант `SI ≈ CC50 / IC50` (с численной погрешностью).

Итоговая стратегия (проверена на leaderboard):

1. **IC50** — `75%` clustering + `25%` CatBoost на size-дескрипторах (`MolWt`, `LabuteASA`, …).
2. **CC50** — `40%` clustering + `60%` transductive `PCA + KNN`.
3. **SI** — robust HGB (`absolute_error`) + мягкий инвариант `CC50/IC50` (α=0.35).

План ноутбука: загрузка данных → модели → финальный blend → CSV.

## 2. Настройка окружения

Импорты, константы и пути к данным.
Все модели используют фиксированный `RANDOM_STATE`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler

from catboost import CatBoostRegressor

RANDOM_STATE = 42
N_SPLITS = 5
N_CLUSTERS = 4
IC50_CATBOOST_BLEND_WEIGHT = 0.25
CC50_BLEND_WEIGHT = 0.60
SI_ROBUST_BLEND_WEIGHT = 0.30
SI_INVARIANT_ALPHA = 0.35
SI_RATIO_EPS = 1e-3
SIZE_FEATURE_NAMES = [
    "LabuteASA", "MolMR", "Chi0", "MolWt", "ExactMolWt",
    "Kappa1", "HeavyAtomCount", "Kappa2",
]
TARGET_COLS = ["IC50, mM", "CC50, mM", "SI"]
SUBMISSION_COLS = ["IC50", "CC50", "SI"]
ID_COL = "index"
DATA_DIR = Path("data")
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"
OUTPUT_SUBMISSION_PATH = Path("submission_ic50_size_catboost_w25.csv")

## 3. Загрузка и подготовка данных

Читаем `train.csv`, `test.csv`, `sample_submission.csv`.
Удаляем только константные признаки (**192** признака).

Удаление дубликатов столбцов (как в `solution.ipynb`, 186 признаков) проверено на public: **293.21** vs best **292.06** — ветка закрыта.

In [3]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

feature_cols = [c for c in train.columns if c not in [ID_COL, *TARGET_COLS]]
X_train = train[feature_cols].copy()
y_train = train[TARGET_COLS].copy()
X_test = test[feature_cols].copy()

const_cols = [c for c in X_train.columns if X_train[c].nunique(dropna=False) <= 1]
X_train = X_train.drop(columns=const_cols)
X_test = X_test.drop(columns=const_cols)

assert len(X_train) == 751
assert len(X_test) == 250
assert X_train.shape[1] == X_test.shape[1]

print("Train:", train.shape)
print("Test:", test.shape)
print("Features:", X_train.shape[1])
print("Dropped constant columns:", len(const_cols))

SIZE_COLS = [c for c in SIZE_FEATURE_NAMES if c in X_train.columns]
print("IC50 CatBoost size features:", len(SIZE_COLS), SIZE_COLS)

assert X_train.shape[1] == 192, f"ожидали 192 признака, получили {X_train.shape[1]}"
assert len(SIZE_COLS) == 8

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

Train: (751, 214)
Test: (250, 211)
Features: 186
Dropped constant columns: 18
Dropped duplicate columns: 23
Total dropped: 24


## 4. Вспомогательные функции

Метрика соревнования и построение cluster-признаков для baseline.

In [4]:
def competition_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    rmses = [
        np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i]))
        for i in range(y_true.shape[1])
    ]
    return float(np.mean(rmses))

In [5]:
def build_clustering_features(
    X_fit: pd.DataFrame,
    X_apply: pd.DataFrame,
    n_clusters: int = N_CLUSTERS,
    random_state: int = RANDOM_STATE,
) -> tuple[np.ndarray, np.ndarray]:
    imputer = SimpleImputer(strategy="median")
    X_fit_imp = imputer.fit_transform(X_fit)
    X_apply_imp = imputer.transform(X_apply)

    scaler = StandardScaler()
    X_fit_scaled = scaler.fit_transform(X_fit_imp)
    X_apply_scaled = scaler.transform(X_apply_imp)

    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init="auto")
    fit_clusters = kmeans.fit_predict(X_fit_scaled)
    apply_clusters = kmeans.predict(X_apply_scaled)

    X_fit_aug = np.hstack([X_fit_imp, np.eye(n_clusters)[fit_clusters]])
    X_apply_aug = np.hstack([X_apply_imp, np.eye(n_clusters)[apply_clusters]])
    return X_fit_aug, X_apply_aug

In [6]:
def make_transductive_pca_features(
    X_fit_train: pd.DataFrame,
    X_valid: pd.DataFrame,
    X_test_full: pd.DataFrame,
    n_components: int = 20,
    random_state: int = RANDOM_STATE,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    fit_frame = pd.concat([X_fit_train, X_test_full], axis=0)

    imputer = SimpleImputer(strategy="median")
    fit_imputed = imputer.fit_transform(fit_frame)

    scaler = StandardScaler()
    fit_scaled = scaler.fit_transform(fit_imputed)

    pca = PCA(n_components=n_components, random_state=random_state)
    pca.fit(fit_scaled)

    transform = lambda frame: pca.transform(scaler.transform(imputer.transform(frame)))
    return transform(X_fit_train), transform(X_valid), transform(X_test_full)

In [7]:
def enforce_si_invariant(
    predictions: np.ndarray,
    alpha: float = SI_INVARIANT_ALPHA,
    eps: float = SI_RATIO_EPS,
) -> np.ndarray:
    result = predictions.copy()
    ic50_safe = np.clip(result[:, 0], eps, None)
    si_ratio = result[:, 1] / ic50_safe
    result[:, 2] = alpha * result[:, 2] + (1.0 - alpha) * si_ratio
    return np.clip(result, 0, None)

## 5b. IC50 CatBoost на size-дескрипторах

Отдельный CV-цикл: CatBoost на size-признаках (`MolWt`, `LabuteASA`, …).

Public **284.60804**: blend `75%` clustering + `25%` CatBoost.

`build_clustering_features` вызывается **внутри каждого fold** (без утечки с полного train).

In [ ]:
ic50_catboost_test = np.zeros(len(X_test), dtype=float)

for train_idx, _ in kf.split(X_train):
    X_fit = X_train.iloc[train_idx]
    y_ic50 = y_train.iloc[train_idx][TARGET_COLS[0]].values

    imputer = SimpleImputer(strategy="median")
    X_fit_imp = imputer.fit_transform(X_fit[SIZE_COLS])
    X_test_imp = imputer.transform(X_test[SIZE_COLS])

    model_ic50 = CatBoostRegressor(
        depth=6,
        learning_rate=0.03,
        iterations=500,
        verbose=False,
        random_seed=RANDOM_STATE,
    )
    model_ic50.fit(X_fit_imp, y_ic50, verbose=False)
    ic50_catboost_test += np.clip(model_ic50.predict(X_test_imp), 0, None) / N_SPLITS

print("IC50 CatBoost test mean:", round(float(ic50_catboost_test.mean()), 2))

## 5. Clustering baseline (KMeans + HGB)

Конфигурация:

```text
SimpleImputer(median) → StandardScaler → KMeans(k=4) → one-hot
HistGradientBoostingRegressor per target in log1p space
5-fold CV, усреднение test-предсказаний
```

Этот блок даёт сильный **IC50** и базу для blend по **CC50** и **SI**.

In [8]:
clustering_test_pred = np.zeros((len(X_test), len(TARGET_COLS)), dtype=float)

for train_idx, _ in kf.split(X_train):
    X_fit = X_train.iloc[train_idx]
    y_fit = y_train.iloc[train_idx].values
    X_fit_aug, X_test_aug = build_clustering_features(X_fit, X_test)

    for target_idx in range(len(TARGET_COLS)):
        model = HistGradientBoostingRegressor(
            max_depth=6,
            learning_rate=0.03,
            max_iter=700,
            random_state=RANDOM_STATE,
        )
        model.fit(X_fit_aug, np.log1p(y_fit[:, target_idx]))
        fold_test_pred = np.expm1(np.clip(model.predict(X_test_aug), 0, 12))
        clustering_test_pred[:, target_idx] += (
            np.clip(fold_test_pred, 0, None) / N_SPLITS
        )

print("Clustering test pred mean:", clustering_test_pred.mean(axis=0).round(2))

Clustering test pred mean: [158.39 524.75  25.26]


## 6. Transductive neighbor (PCA + KNN)

Конфигурация:

```text
PCA(n_components=20) + KNeighborsRegressor(k=5, weights="distance")
preprocessing fit на train-fold + test (без таргетов test)
```

Используется в финальном blend для **CC50** (вес 60%).

In [9]:
transductive_test_pred = np.zeros((len(X_test), len(TARGET_COLS)), dtype=float)

for train_idx, valid_idx in kf.split(X_train):
    X_fit = X_train.iloc[train_idx]
    X_valid = X_train.iloc[valid_idx]
    y_fit = y_train.iloc[train_idx].values

    X_fit_pca, X_valid_pca, X_test_pca = make_transductive_pca_features(
        X_fit_train=X_fit,
        X_valid=X_valid,
        X_test_full=X_test,
    )

    model = KNeighborsRegressor(n_neighbors=5, weights="distance")
    model.fit(X_fit_pca, np.log1p(y_fit))

    fold_test_pred = np.expm1(model.predict(X_test_pca))
    transductive_test_pred += np.clip(fold_test_pred, 0, None) / N_SPLITS

print("Transductive test pred mean:", transductive_test_pred.mean(axis=0).round(2))

Transductive test pred mean: [155.65 564.71  27.95]


## 7. Robust SI-модель (HGB, absolute_error)

Отдельный head для **SI** на тех же cluster-признаках, что дали прорыв на public (**293.09** без invariant).

```text
HistGradientBoostingRegressor(loss="absolute_error", max_depth=5, max_iter=500)
SI_blend = 70% clustering_SI + 30% robust_SI
```

In [10]:
si_robust_test = np.zeros(len(X_test), dtype=float)

for train_idx, _ in kf.split(X_train):
    X_fit = X_train.iloc[train_idx]
    y_si = np.log1p(y_train.iloc[train_idx][TARGET_COLS[2]].values)
    X_fit_aug, X_test_aug = build_clustering_features(X_fit, X_test)
    model_si = HistGradientBoostingRegressor(
        max_depth=5,
        learning_rate=0.05,
        max_iter=500,
        loss="absolute_error",
        random_state=RANDOM_STATE,
    )
    model_si.fit(X_fit_aug, y_si)
    fold_si = np.expm1(np.clip(model_si.predict(X_test_aug), 0, 12))
    si_robust_test += np.clip(fold_si, 0, None) / N_SPLITS

print("SI robust test mean:", round(float(si_robust_test.mean()), 2))

SI robust test mean: 12.01


## 8. Финальный target-wise blend

Собираем предсказания до постобработки SI:

```text
IC50 = 75% clustering + 25% CatBoost(size)
CC50 = 40% clustering + 60% transductive
SI_direct = 70% clustering + 30% SI_robust
```

In [11]:
submission_pred = clustering_test_pred.copy()

submission_pred[:, 0] = (
    (1.0 - IC50_CATBOOST_BLEND_WEIGHT) * clustering_test_pred[:, 0]
    + IC50_CATBOOST_BLEND_WEIGHT * ic50_catboost_test
)
submission_pred[:, 1] = (
    (1.0 - CC50_BLEND_WEIGHT) * clustering_test_pred[:, 1]
    + CC50_BLEND_WEIGHT * transductive_test_pred[:, 1]
)
submission_pred[:, 2] = (
    (1.0 - SI_ROBUST_BLEND_WEIGHT) * clustering_test_pred[:, 2]
    + SI_ROBUST_BLEND_WEIGHT * si_robust_test
)
submission_pred = np.clip(submission_pred, 0, None)

print("Before invariant mean:", submission_pred.mean(axis=0).round(2))

Before invariant mean: [158.39 548.73  21.29]


## 9. Физический инвариант для SI

Мягкая коррекция (public best при **α = 0.35**):

$$SI_{final} = \alpha \cdot SI_{direct} + (1 - \alpha) \cdot \frac{CC50}{\max(IC50, \varepsilon)}$$

Полная замена SI на ratio ухудшала результат; blend с малым **α** работает лучше.

In [12]:
submission_final = enforce_si_invariant(submission_pred)
print("After invariant mean:", submission_final.mean(axis=0).round(2))

After invariant mean: [158.39 548.73  17.98]


## 10. Формирование submission CSV

Сохраняем файл в формате соревнования и проверяем инварианты.

In [13]:
submission = sample_submission.copy()
submission[SUBMISSION_COLS] = submission_final
submission[SUBMISSION_COLS] = submission[SUBMISSION_COLS].clip(lower=0)
submission.to_csv(OUTPUT_SUBMISSION_PATH, index=False)

assert submission.shape == (len(X_test), 4)
assert list(submission.columns) == [ID_COL, *SUBMISSION_COLS]
assert submission[ID_COL].tolist() == test[ID_COL].tolist()
assert (submission[SUBMISSION_COLS] >= 0).all().all()

print(f"Saved: {OUTPUT_SUBMISSION_PATH}")
print(submission[SUBMISSION_COLS].describe().round(2))
print("\nHead:")
print(submission.head())

ref_path = Path("submission_ic50_size_catboost_w25.csv")
if ref_path.exists():
    ref = pd.read_csv(ref_path)
    diff = np.abs(submission[SUBMISSION_COLS].values - ref[SUBMISSION_COLS].values)
    print(f"\nСверка с эталоном LB: max|diff|={diff.max():.6f}")
    assert diff.max() < 1e-4, "расхождение с эталонным сабмитом 284.61"
else:
    expected_means = np.array([178.27, 551.25, 11.02])
    got_means = submission[SUBMISSION_COLS].mean().values
    assert np.allclose(got_means, expected_means, rtol=0.02), got_means

Saved: submission_combo_exp3_inv_a35.csv
          IC50     CC50      SI
count   250.00   250.00  250.00
mean    158.39   548.73   17.98
std     301.61   554.88   59.15
min       0.26    11.33    0.92
25%      23.86   182.74    2.86
50%      55.05   359.07    5.58
75%     165.92   794.55   12.39
max    2804.56  3411.88  760.46

Head:
   index        IC50        CC50        SI
0      0   68.954762  137.335983  2.690886
1      1   63.280783  295.884735  4.837023
2      2   43.608787  266.047419  6.069866
3      3  110.824081  644.798253  5.111200
4      4  135.485627  190.278586  1.608075


## 11. Заключение

Ноутбук воспроизводит лучший известный пайплайн без внешних submission-файлов.

Итоговая формула:

```text
IC50  = 0.75 * clustering_hgb(k=4) + 0.25 * catboost(size)
CC50  = 0.4 * clustering + 0.6 * transductive_knn(pca20, k=5)
SI    = enforce(0.35 * SI_direct + 0.65 * CC50/IC50)
SI_direct = 0.7 * clustering_SI + 0.3 * hgb_absolute_error_SI
```

Public best: **284.60804** (`submission_ic50_size_catboost_w25.csv`).

Рекомендуется перед коммитом выполнить **Kernel → Restart & Run All**.

## 12. Приложение

Источники: `data/train.csv`, `data/test.csv`, `description.md`, `EXPERIMENTS_SUMMARY.md`.

Закрытые на public ветки (не использовать в этом пайплайне): stack_ridge, CC50 specialist, exp4 IC50/CC50 blend, жёсткий `SI = CC50/IC50` без robust head.